# EIN XML Person-Level Parser

This notebook parses IRS XML filings into a **person-level** pandas DataFrame. Each row represents one person found in a filing, while the nonprofit's EIN, name, website, and address are repeated on that row so the data is ready for downstream analysis like title filtering and email guessing.


## Interview Explanation

Here is the logic in plain English:

1. We locate the XML files from the nonprofit filings folders.
2. We parse one XML file at a time using Python's built-in XML parser.
3. For each file, we extract organization-level metadata once, such as EIN, organization name, website, phone number, address, and tax year.
4. Then we search the same file for person-related groups like officers, board members, and key employees.
5. We flatten the nested XML into tabular rows by combining the organization metadata with each person found in that filing.
6. We store the rows in a list of dictionaries and then convert that list into a pandas DataFrame.
7. We keep one row per person because that makes it much easier to analyze titles and to guess individual email patterns later.

Why repeat EIN and organization data on every person row? Because this is a denormalized analysis table. The duplication is intentional: it makes filtering, grouping, exporting, and later merges much simpler in pandas without needing another join every time.


In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd


In [2]:
# XML files live in these folders in this repo.
XML_FOLDERS = [
    Path("../XML"),
    Path("../data/2025_XML"),
]

xml_files = []
for folder in XML_FOLDERS:
    if folder.exists():
        xml_files.extend(sorted(folder.glob("*.xml")))

# Remove duplicates by filename while keeping order, because the same filing
# can appear in both ../XML and ../data/2025_XML.
deduped_xml_files = []
seen_file_names = set()
for xml_file in xml_files:
    if xml_file.name in seen_file_names:
        continue
    seen_file_names.add(xml_file.name)
    deduped_xml_files.append(xml_file)

xml_files = deduped_xml_files

print(f"Found {len(xml_files):,} XML files across {len(XML_FOLDERS)} folders.")
xml_files[:5]


Found 2,747 XML files across 2 folders.


[PosixPath('../XML/202502349349301450_public.xml'),
 PosixPath('../XML/202502389349100745_public.xml'),
 PosixPath('../XML/202502389349300135_public.xml'),
 PosixPath('../XML/202502389349300600_public.xml'),
 PosixPath('../XML/202502399349100520_public.xml')]

In [3]:
IRS_NS = {"irs": "http://www.irs.gov/efile"}
PEOPLE_GROUP_TAGS = [
    "BusinessOfficerGrp",
    "Form990PartVIISectionAGrp",
    "OfficerDirTrstKeyEmplGrp",
]


def get_text(node, path, ns=IRS_NS):
    """Return stripped text for the first matching child, or None if missing."""
    if node is None:
        return None

    found = node.find(path, ns)
    if found is None or found.text is None:
        return None

    text = found.text.strip()
    return text or None


def first_non_null(*values):
    for value in values:
        if value is not None:
            return value
    return None


def extract_business_name(node):
    if node is None:
        return None

    line1 = get_text(node, "irs:BusinessNameLine1Txt")
    line2 = get_text(node, "irs:BusinessNameLine2Txt")
    parts = [part for part in [line1, line2] if part]
    return " | ".join(parts) if parts else None


def extract_address_fields(node):
    if node is None:
        return {
            "address_line1": None,
            "city": None,
            "state": None,
            "zip": None,
            "full_address": None,
        }

    address_line1 = get_text(node, "irs:AddressLine1Txt")
    city = get_text(node, "irs:CityNm")
    state = get_text(node, "irs:StateAbbreviationCd")
    zip_code = get_text(node, "irs:ZIPCd")

    full_address = ", ".join([part for part in [address_line1, city, state, zip_code] if part]) or None

    return {
        "address_line1": address_line1,
        "city": city,
        "state": state,
        "zip": zip_code,
        "full_address": full_address,
    }


def extract_org_metadata(root, source_file):
    filer = root.find(".//irs:Filer", IRS_NS)
    irs_990 = root.find(".//irs:IRS990", IRS_NS)

    org_name = first_non_null(
        extract_business_name(filer.find("irs:BusinessName", IRS_NS) if filer is not None else None),
        extract_business_name(irs_990.find("irs:BusinessName", IRS_NS) if irs_990 is not None else None),
    )

    org_phone = first_non_null(
        get_text(filer, "irs:PhoneNum"),
        get_text(irs_990, "irs:PhoneNum"),
    )

    address_node = None
    if filer is not None:
        address_node = first_non_null(
            filer.find("irs:USAddress", IRS_NS),
            filer.find("irs:ForeignAddress", IRS_NS),
        )
    if address_node is None and irs_990 is not None:
        address_node = first_non_null(
            irs_990.find("irs:USAddress", IRS_NS),
            irs_990.find("irs:ForeignAddress", IRS_NS),
        )

    address_fields = extract_address_fields(address_node)

    return {
        "source_file": source_file.name,
        "ein": get_text(filer, "irs:EIN") if filer is not None else None,
        "org_name": org_name,
        "website": get_text(root, ".//irs:WebsiteAddressTxt"),
        "phone": org_phone,
        "tax_year": get_text(root, ".//irs:TaxYr"),
        **address_fields,
    }


def extract_person_record(person_node, group_tag):
    person_name = first_non_null(
        get_text(person_node, "irs:PersonNm"),
        extract_business_name(person_node.find("irs:BusinessName", IRS_NS)),
    )

    person_title = first_non_null(
        get_text(person_node, "irs:PersonTitleTxt"),
        get_text(person_node, "irs:TitleTxt"),
    )

    person_hours = first_non_null(
        get_text(person_node, "irs:AverageHoursPerWeekRt"),
        get_text(person_node, "irs:AverageHrsPerWkDevotedToPosRt"),
    )

    role_type_map = {
        "BusinessOfficerGrp": "business_officer",
        "Form990PartVIISectionAGrp": "form_990_section_a",
        "OfficerDirTrstKeyEmplGrp": "officer_director_trustee_key_employee",
    }

    return {
        "person_name": person_name,
        "person_title": person_title,
        "person_hours_per_week": person_hours,
        "person_role_type": role_type_map.get(group_tag, group_tag),
    }


def parse_xml_file(xml_path):
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return [{
            "source_file": xml_path.name,
            "ein": None,
            "org_name": None,
            "website": None,
            "phone": None,
            "address_line1": None,
            "city": None,
            "state": None,
            "zip": None,
            "full_address": None,
            "tax_year": None,
            "person_name": None,
            "person_title": None,
            "person_hours_per_week": None,
            "person_role_type": "parse_error",
        }]

    org_data = extract_org_metadata(root, xml_path)
    rows = []

    for group_tag in PEOPLE_GROUP_TAGS:
        for person_node in root.findall(f".//irs:{group_tag}", IRS_NS):
            person_data = extract_person_record(person_node, group_tag)
            rows.append({**org_data, **person_data})

    if not rows:
        rows.append({
            **org_data,
            "person_name": None,
            "person_title": None,
            "person_hours_per_week": None,
            "person_role_type": None,
        })

    return rows


In [4]:
records = []
for xml_file in xml_files:
    records.extend(parse_xml_file(xml_file))

xml_people_df = pd.DataFrame(records)

# Keep EIN as string-like data so we do not lose formatting during later merges.
xml_people_df["ein"] = xml_people_df["ein"].astype("string")

xml_people_df.head()


,source_file,ein,org_name,website,phone,tax_year,address_line1,city,state,zip,full_address,person_name,person_title,person_hours_per_week,person_role_type
0,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG,8584854800,2024,13010 Paseo Lucido,San Diego,CA,92128,"13010 Paseo Lucido, San Diego, CA, 92128",Denise Davis,Treasurer,NaN,business_officer
1,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG,8584854800,2024,13010 Paseo Lucido,San Diego,CA,92128,"13010 Paseo Lucido, San Diego, CA, 92128",Kurt Trecker,President,0.87,form_990_section_a
2,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG,8584854800,2024,13010 Paseo Lucido,San Diego,CA,92128,"13010 Paseo Lucido, San Diego, CA, 92128",Jill Trecker,Vice President,0.83,form_990_section_a
3,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG,8584854800,2024,13010 Paseo Lucido,San Diego,CA,92128,"13010 Paseo Lucido, San Diego, CA, 92128",Rhonda Ewald,Secretary,0.63,form_990_section_a
4,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG,8584854800,2024,13010 Paseo Lucido,San Diego,CA,92128,"13010 Paseo Lucido, San Diego, CA, 92128",Denise Davis,Treasurer,5.37,form_990_section_a


In [5]:
print(f"Row count: {len(xml_people_df):,}")
print(f"Unique EINs: {xml_people_df['ein'].nunique(dropna=True):,}")
print(f"Rows with a person name: {xml_people_df['person_name'].notna().sum():,}")
print("\nColumns:")
print(list(xml_people_df.columns))


Row count: 19,723
Unique EINs: 2,485
Rows with a person name: 19,723

Columns:
['source_file', 'ein', 'org_name', 'website', 'phone', 'tax_year', 'address_line1', 'city', 'state', 'zip', 'full_address', 'person_name', 'person_title', 'person_hours_per_week', 'person_role_type']


## Sanity Checks

These checks make the notebook easier to defend in an interview because they show that the parser was validated instead of being treated like a black box.


In [6]:
# 1. Example EIN with multiple people should produce multiple rows.
multi_person_examples = (
    xml_people_df.groupby('ein', dropna=True)
    .size()
    .sort_values(ascending=False)
    .head(10)
)
multi_person_examples


ein
203049742    245
951184680     99
952700856     89
330849518     78
330496646     78
952406199     77
953699122     72
951691313     61
330430474     61
260216910     59
dtype: int64

In [7]:
# 2. Confirm website extraction works.
xml_people_df.loc[xml_people_df['website'].notna(), ['source_file', 'ein', 'org_name', 'website']].head(10)


,source_file,ein,org_name,website
0,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
1,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
2,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
3,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
4,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
5,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
6,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
7,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
8,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG
9,202502349349301450_public.xml,330427423,Rancho Bernardo High School Foundation,WWW.RBHSFOUNDATION.ORG


In [8]:
# 3. Confirm fallback rows exist for filings with no extracted people.
xml_people_df.loc[xml_people_df['person_name'].isna(), ['source_file', 'ein', 'org_name', 'website']].head(10)


,source_file,ein,org_name,website


## How To Explain The Design

If an interviewer asks why the same EIN appears on multiple rows, the answer is: **that is expected because the table is intentionally built at person grain, not organization grain.** One organization can have many people attached to the same filing, so the organization metadata is copied onto each person row. That makes later analysis much simpler, especially if you want to guess one email per person or filter for roles like President, CEO, Treasurer, or Board Chair.

If you later want an organization-level table, you can always aggregate this person-level DataFrame back up by EIN. It is much easier to collapse a detailed table than to recover missing person detail from an organization-only table.


In [9]:

output_path = Path('xml_people_df.csv')
xml_people_df.to_csv(output_path, index=False)
output_path


PosixPath('xml_people_df.csv')